# 02. Employment Contract Dataset Filtering & Taxonomy Mapping

## Executive Summary & Objectives
This notebook refines the raw **CUAD (Contract Understanding Atticus Dataset)** corpus into a specialized **Employment & Work Agreement Dataset** designed specifically for **ClauseGuard**.

### Why Filter the Dataset?
1. **Domain Focus:** ClauseGuard analyzes **Employment Contracts** (Employer–Employee agreements). Commercial B2B categories like *Freight/Incoterms*, *Source Code Escrow*, or *Volume Restrictions* add unnecessary noise.
2. **10 Core Employment Categories:** We map the 41 broad CUAD categories into **10 targeted employment-relevant classes** (e.g., `non_compete`, `no_solicit_employees`, `termination_notice`, `ip_assignment`, `confidentiality`, `governing_law`).
3. **Model Performance:** Training LegalBERT on a clean, focused 10-class taxonomy yields significantly higher classification accuracy, macro-F1, and faster inference than a noisy 41-class setup.
4. **Zero Document Leakage:** We enforce document-grouped splits (`GroupShuffleSplit` on `contract_title`) to prevent provisions from the same contract from appearing in both training and test sets.

## 1. Setup & Dependencies
We import necessary standard data science libraries (`pandas`, `numpy`, `matplotlib`, `seaborn`) and `GroupShuffleSplit` for zero-leakage dataset partitioning.

In [ ]:
import os
import sys
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

# Display formatting
pd.set_option('display.max_colwidth', 150)
pd.set_option('display.max_columns', 20)
sns.set_theme(style="whitegrid")

print("Python Version:", sys.version)
print("Pandas Version:", pd.__version__)

## 2. Load Raw Master Extracted CUAD Clauses
We load the clean extracted clauses CSV generated in `01_cuad_exploration.ipynb` (`ml/data/cuad_processed/full_clean_clauses.csv`).

In [ ]:
# Define file paths relative to project root
PROJECT_ROOT = Path("..").resolve()
MASTER_CLAUSES_PATH = PROJECT_ROOT / "ml" / "data" / "cuad_processed" / "full_clean_clauses.csv"

if not MASTER_CLAUSES_PATH.exists():
    # Fallback to local path
    MASTER_CLAUSES_PATH = Path("ml/data/cuad_processed/full_clean_clauses.csv")

print(f"[*] Loading Master Extracted CUAD Clauses from: {MASTER_CLAUSES_PATH.resolve()}...")
df_master = pd.read_csv(MASTER_CLAUSES_PATH)
print(f"[✓] Master dataset loaded successfully! Total records: {len(df_master):,}")
print(f"Unique source contracts: {df_master['contract_title'].nunique()}")
print(f"Unique raw categories: {df_master['category'].nunique()}")

df_master.head(3)

## 3. Define Employment Taxonomy Mapping (`CUAD_TO_EMPLOYMENT_MAP`)
Here we explicitly map broad CUAD legal categories to ClauseGuard's target **10 Employment Classes**:

| Target Employment Category | Mapped Source CUAD Categories | Purpose in Employment Contracts |
| :--- | :--- | :--- |
| **`non_compete`** | `Non-Compete` | Restrictions on working for competitors after employment |
| **`no_solicit_employees`** | `No-Solicit Of Employees` | Anti-poaching restrictions on hiring former colleagues |
| **`no_solicit_customers`** | `No-Solicit Of Customers` | Restrictions on soliciting company clients/customers |
| **`termination_notice`** | `Termination For Convenience`, `Notice Period To Terminate Renewal` | Notice periods required for contract termination |
| **`ip_assignment`** | `Ip Ownership Assignment`, `Joint Ip Ownership` | Work-for-hire invention & IP assignment clauses |
| **`confidentiality`** | `Non-Disparagement`, `Confidentiality` | Trade secret protection & non-disparagement obligations |
| **`governing_law`** | `Governing Law` | Jurisdiction and applicable state/country law |
| **`exclusivity`** | `Exclusivity` | Moonlighting and exclusive service commitments |
| **`liability_indemnity`** | `Cap On Liability`, `Uncapped Liability` | Damage caps and indemnity provisions |
| **`other_general`** | Standard non-annotated text | Baseline non-risk paragraph text |

In [ ]:
CUAD_TO_EMPLOYMENT_MAP = {
    "Non-Compete": "non_compete",
    "No-Solicit Of Employees": "no_solicit_employees",
    "No-Solicit Of Customers": "no_solicit_customers",
    "Termination For Convenience": "termination_notice",
    "Notice Period To Terminate Renewal": "termination_notice",
    "Ip Ownership Assignment": "ip_assignment",
    "Joint Ip Ownership": "ip_assignment",
    "Non-Disparagement": "confidentiality",
    "Governing Law": "governing_law",
    "Exclusivity": "exclusivity",
    "Cap On Liability": "liability_indemnity",
    "Uncapped Liability": "liability_indemnity"
}

print("Configured Target Categories:", len(set(CUAD_TO_EMPLOYMENT_MAP.values())))
print("Explicit Mapping Dictionary:")
for cuad_cat, emp_cat in CUAD_TO_EMPLOYMENT_MAP.items():
    print(f" - '{cuad_cat}' -> '{emp_cat}'")

## 4. Filter & Map Dataset
We apply the taxonomy dictionary to filter out unrelated B2B categories (like Incoterms or Freight) and attach the clean `employment_category` label to each row.

In [ ]:
# Filter dataset to only mapped rows
df_filtered = df_master[df_master["category"].isin(CUAD_TO_EMPLOYMENT_MAP.keys())].copy()
df_filtered["employment_category"] = df_filtered["category"].map(CUAD_TO_EMPLOYMENT_MAP)

print(f"Initial raw clauses: {len(df_master):,}")
print(f"Filtered Employment clauses: {len(df_filtered):,}")
print(f"Filtered out {len(df_master) - len(df_filtered):,} unrelated B2B clauses.")

print("\nDistribution by New Employment Category:")
print(df_filtered["employment_category"].value_counts())

## 5. Add Negative / General Contract Paragraphs (`other_general`)
To give LegalBERT a **"None of the Above / Standard Boilerplate"** class, we extract unannotated text gaps from `data/CUADv1.json` (paragraphs/sections that fall outside all 41 legal question answer spans) and label them as **`other_general`**.

In [ ]:
CUAD_MASTER_PATH = PROJECT_ROOT / "data" / "CUADv1.json"
if not CUAD_MASTER_PATH.exists():
    CUAD_MASTER_PATH = Path("data/CUADv1.json")

negative_samples = []
if CUAD_MASTER_PATH.exists():
    print(f"[*] Extracting unannotated text gaps (`other_general`) from: {CUAD_MASTER_PATH}...")
    with open(CUAD_MASTER_PATH, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
        
    for doc in raw_data.get("data", []):
        title = doc.get("title", "")
        for para in doc.get("paragraphs", []):
            context = para.get("context", "")
            
            # 1. Collect all positive answer character ranges in this contract
            spans = []
            for qa in para.get("qas", []):
                for ans in qa.get("answers", []):
                    st = ans.get("answer_start", -1)
                    txt = ans.get("text", "")
                    if st >= 0 and txt:
                        spans.append((st, st + len(txt)))
            
            spans = sorted(spans, key=lambda x: x[0])
            
            # 2. Extract unannotated text gaps between positive answer spans
            last_end = 0
            for st, end in spans:
                if st > last_end + 80:
                    gap_text = context[last_end:st].strip()
                    for snippet in gap_text.split("\n\n"):
                        clean_snip = snippet.strip()
                        if 60 <= len(clean_snip) <= 350 and not clean_snip.startswith("Source:") and not clean_snip.startswith("EXHIBIT"):
                            negative_samples.append({
                                "contract_title": title,
                                "category": "General / Unannotated",
                                "clause_text": clean_snip,
                                "start_char": -1,
                                "end_char": -1,
                                "employment_category": "other_general"
                            })
                if end > last_end:
                    last_end = end

df_negatives = pd.DataFrame(negative_samples).drop_duplicates(subset=["clause_text"])
print(f"[✓] Extracted {len(df_negatives):,} clean negative/general clauses (`other_general`).")

# Combine positive and negative clauses
df_employment = pd.concat([df_filtered, df_negatives], ignore_index=True)
print(f"Total Combined Dataset Size: {len(df_employment):,} clauses.")

## 6. Clean Text & Deduplicate Clauses
We strip extra whitespace, remove short text artifacts (< 15 characters), and remove exact duplicate `(clause_text, employment_category)` entries.

In [ ]:
initial_len = len(df_employment)
df_employment["clause_text"] = df_employment["clause_text"].str.replace(r"\s+", " ", regex=True).str.strip()

# Filter out tiny noise
df_employment = df_employment[df_employment["clause_text"].str.len() >= 15].copy()

# Deduplicate exact pairs
df_employment = df_employment.drop_duplicates(subset=["clause_text", "employment_category"]).copy()
final_len = len(df_employment)

print(f"Initial dataset size: {initial_len:,}")
print(f"Final clean & deduplicated dataset size: {final_len:,}")
print(f"Removed duplicate/noise rows: {initial_len - final_len:,}")

## 7. Visualize Employment Category Distribution
We inspect the final class breakdown across all 10 employment categories.

In [ ]:
class_counts = df_employment["employment_category"].value_counts()

print("=== Employment Category Breakdown ===")
for cls, count in class_counts.items():
    print(f" - {cls:22s}: {count:5,d} clauses ({count/len(df_employment):.1%})")

plt.figure(figsize=(12, 6))
sns.barplot(x=class_counts.values, y=class_counts.index, palette="magma")
plt.title(f"ClauseGuard Employment Dataset (Total: {len(df_employment):,} Clauses across 10 Classes)")
plt.xlabel("Number of Extracted Clauses")
plt.ylabel("Employment Category")
plt.tight_layout()
plt.show()

## 8. Create Numeric Label Mappings (`label2id` & `id2label`)
We sort category names deterministically and assign integer label IDs `0..9` for PyTorch/LegalBERT training.

In [ ]:
sorted_categories = sorted(df_employment["employment_category"].unique())
label2id = {cat: idx for idx, cat in enumerate(sorted_categories)}
id2label = {idx: cat for idx, cat in enumerate(sorted_categories)}

df_employment["label_id"] = df_employment["employment_category"].map(label2id)

print("Total Classes:", len(label2id))
print("\nNumeric Label Mapping:")
for idx, cat in id2label.items():
    count = (df_employment['label_id'] == idx).sum()
    print(f"  ID {idx}: {cat:22s} ({count:,} clauses)")

## 9. Document-Grouped Train / Validation / Test Splits (0 Data Leakage)
To prevent data leakage, **all provisions from the same source contract are kept strictly within the same split** (`GroupShuffleSplit` on `contract_title`).

- **Train:** 70%
- **Validation:** 15%
- **Test:** 15%

In [ ]:
def verify_zero_leakage(train_docs, val_docs, test_docs):
    tr_set = set(train_docs)
    va_set = set(val_docs)
    te_set = set(test_docs)
    
    overlap_tr_va = tr_set.intersection(va_set)
    overlap_tr_te = tr_set.intersection(te_set)
    overlap_va_te = va_set.intersection(te_set)
    
    print(f"Overlap Train / Val:  {len(overlap_tr_va)} contracts")
    print(f"Overlap Train / Test: {len(overlap_tr_te)} contracts")
    print(f"Overlap Val / Test:   {len(overlap_va_te)} contracts")
    assert len(overlap_tr_va) == 0 and len(overlap_tr_te) == 0 and len(overlap_va_te) == 0, "DATA LEAKAGE DETECTED!"
    print("[✓] Zero Document-Level Leakage Verified!")

# 1. Split off Test set (15%)
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_val_idx, test_idx = next(gss_test.split(df_employment, groups=df_employment["contract_title"]))
df_train_val = df_employment.iloc[train_val_idx].copy()
df_test = df_employment.iloc[test_idx].copy()

# 2. Split Train / Val (85% * 0.1765 = 15% overall for Val, 70% overall for Train)
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)
train_idx, val_idx = next(gss_val.split(df_train_val, groups=df_train_val["contract_title"]))
df_train = df_train_val.iloc[train_idx].copy()
df_val = df_train_val.iloc[val_idx].copy()

print(f"Train Set:      {len(df_train):5,d} clauses ({df_train['contract_title'].nunique()} contracts)")
print(f"Validation Set: {len(df_val):5,d} clauses ({df_val['contract_title'].nunique()} contracts)")
print(f"Test Set:       {len(df_test):5,d} clauses ({df_test['contract_title'].nunique()} contracts)")
print("-" * 60)
verify_zero_leakage(df_train["contract_title"], df_val["contract_title"], df_test["contract_title"])

## 10. Export Processed Datasets
We export the clean CSV files and `label_mapping.json` to `ml/data/employment_processed/`.

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "ml" / "data" / "employment_processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save CSV splits
df_train.to_csv(OUTPUT_DIR / "train.csv", index=False)
df_val.to_csv(OUTPUT_DIR / "val.csv", index=False)
df_test.to_csv(OUTPUT_DIR / "test.csv", index=False)
df_employment.to_csv(OUTPUT_DIR / "employment_full_clauses.csv", index=False)

# Save label mappings
label_map_path = OUTPUT_DIR / "label_mapping.json"
with open(label_map_path, "w", encoding="utf-8") as f:
    json.dump({"id2label": id2label, "label2id": label2id}, f, indent=2)

print(f"[✓] Successfully exported clean employment datasets to: {OUTPUT_DIR.resolve()}")
print("Exported Artifacts:")
print(" - train.csv")
print(" - val.csv")
print(" - test.csv")
print(" - employment_full_clauses.csv")
print(" - label_mapping.json")